<!--
Copyright (c) 2026 Milin Patel
Hochschule Kempten - University of Applied Sciences
Campustage 2026 - Automatisiertes Fahren! Sicher durch Zufall?
MIT License - see LICENSE.
-->


# Der KI-Blick — Wie sieht ein selbstfahrendes Auto?
### Campustage 2026 · *Automatisiertes Fahren! Sicher durch Zufall?*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/00_workshop_der_ki_blick.ipynb)

**Milin Patel · Hochschule Kempten, IFM · MIT Licence**

---

Ein selbstfahrendes Auto erkennt seine Umgebung mit **Kamera** und **KI**.
Hier schaust du der KI beim Sehen zu — und findest heraus, **wo sie versagt.**

Bei jeder Zelle links auf **▶** klicken. In den interaktiven Schritten gibt es **Knöpfe** —
nach jeder Änderung auf den Knopf drücken, dann rechnet die KI neu.


---
## Schritt 0 · Vorbereiten
Einmal ausführen. Lädt die KI (ca. 1 Minute).


In [ ]:
%pip install ultralytics -q
import io, numpy as np, requests
from PIL import Image, ImageDraw, ImageFilter
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
try:
    from google.colab import output as _co; _co.enable_custom_widget_manager()
except Exception: pass
ORANGE=(232,133,12); ROT=(200,60,60)
model=YOLO("yolov8n.pt"); NAMES=model.names
DE={"person":"Mensch","car":"Auto","bus":"Bus","truck":"LKW","bicycle":"Fahrrad",
    "motorcycle":"Motorrad","stop sign":"Stoppschild","traffic light":"Ampel","train":"Zug"}
def de(n): return DE.get(n,n)

def erkenne(img, schwelle):
    res=model.predict(img, conf=schwelle, verbose=False)[0]
    out=img.copy(); d=ImageDraw.Draw(out); personen=0
    for b in res.boxes:
        x1,y1,x2,y2=[int(v) for v in b.xyxy[0]]
        cf=float(b.conf[0]); cid=int(b.cls[0])
        col=ORANGE if cf>=0.5 else ROT
        d.rectangle([x1,y1,x2,y2], outline=col, width=3)
        d.text((x1+4,max(0,y1-12)), f"{de(NAMES.get(cid,''))} {cf:.0%}", fill=col)
        if NAMES.get(cid)=="person": personen+=1
    return out, personen
print("Bereit.")


---
## Schritt 1 · Das Bild

Wir arbeiten das ganze Notebook über mit **einem** Bild.
Standard ist eine Straßenszene.

**Eigenes Foto (optional):** Datei über *Durchsuchen* wählen, dann auf **„Dieses Bild verwenden"** drücken.
Erst dann gilt dein Bild für alle Schritte.


In [ ]:
BILD = Image.open(io.BytesIO(requests.get("https://ultralytics.com/images/bus.jpg",timeout=20).content)).convert("RGB")
up = widgets.FileUpload(accept="image/*", multiple=False, description="Durchsuchen")
btn = widgets.Button(description="Dieses Bild verwenden", button_style="primary")
out1 = widgets.Output()

def _vorschau(msg):
    with out1:
        clear_output(wait=True)
        plt.figure(figsize=(9,6)); plt.imshow(BILD); plt.axis("off"); plt.title(msg); plt.show()

def _use(_):
    global BILD
    if up.value:
        v=up.value
        data=(list(v.values())[0]["content"] if isinstance(v,dict) else v[0]["content"])
        BILD=Image.open(io.BytesIO(data)).convert("RGB")
        _vorschau("Dein Bild ist geladen — es gilt jetzt für alle Schritte.")
    else:
        _vorschau("Kein Foto gewählt — es gilt weiter das Standardbild.")

btn.on_click(_use)
display(widgets.HBox([up, btn]), out1)
_vorschau("Standardbild. (Optional eigenes wählen und Knopf drücken.)")


---
## Schritt 2 · Du zählst zuerst

Schau das Bild oben genau an: **Wie viele Menschen siehst du?** Merk dir die Zahl.


---
## Schritt 3 · Jetzt schaut die KI

Ausführen — die KI zeichnet Boxen mit **Konfidenz** und zählt die Menschen.
Gleich viele wie du?


In [ ]:
img,n=erkenne(BILD,0.25)
plt.figure(figsize=(9,6)); plt.imshow(img); plt.axis("off")
plt.title(f"Die KI erkennt {n} Menschen"); plt.show()


---
## Schritt 4 · Die Grenze entscheidet

Die KI ist sich **nie 100 % sicher**. Ein Mensch legt fest, ab welcher Konfidenz etwas als *erkannt* gilt: die **Schwelle.**

Zuerst der direkte Vergleich — links niedrig, rechts hoch. Achte, wie rechts **Menschen verschwinden.**


In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,6))
for a,s in zip(ax,(0.25,0.70)):
    im,k=erkenne(BILD,s); a.imshow(im); a.axis("off"); a.set_title(f"Schwelle {s:.2f}  →  {k} Menschen")
plt.tight_layout(); plt.show()


**Jetzt du:** Schwelle einstellen und auf **„Erkennen"** drücken. Bei jedem Druck rechnet die KI neu.


In [ ]:
s_slider=widgets.FloatSlider(min=0.05,max=0.95,step=0.05,value=0.25,description="Schwelle",continuous_update=False)
s_btn=widgets.Button(description="Erkennen", button_style="primary")
s_out=widgets.Output()
def _run_s(_):
    with s_out:
        clear_output(wait=True)
        im,k=erkenne(BILD, s_slider.value)
        plt.figure(figsize=(9,6)); plt.imshow(im); plt.axis("off")
        plt.title(f"Schwelle {s_slider.value:.2f}  →  {k} Menschen erkannt"); plt.show()
s_btn.on_click(_run_s)
display(widgets.HBox([s_slider, s_btn]), s_out); _run_s(None)


---
## Schritt 5 · Die Welt ist nicht sauber

Dasselbe Bild bei **Klar, Nebel, Regen** — gleiche Schwelle. Sieh, wie die Zahl **einbricht.**


In [ ]:
def mach_wetter(img, art):
    if art=="Nebel":
        n=Image.new("RGB",img.size,(205,205,210)); return Image.blend(img,n,0.55).filter(ImageFilter.GaussianBlur(2))
    if art=="Regen":
        o=img.copy(); d=ImageDraw.Draw(o); rng=np.random.default_rng(0)
        for _ in range(500):
            x=int(rng.integers(0,img.width)); y=int(rng.integers(0,img.height)); d.line([(x,y),(x-2,y+12)],fill=(185,190,205))
        return Image.eval(o, lambda p:int(p*0.8))
    return img
fig,ax=plt.subplots(1,3,figsize=(16,5))
for a,art in zip(ax,["Klar","Nebel","Regen"]):
    im,k=erkenne(mach_wetter(BILD,art),0.25); a.imshow(im); a.axis("off"); a.set_title(f"{art}  →  {k} Menschen")
plt.tight_layout(); plt.show()


---
## Das nimmst du mit

> **Die KI sieht nicht, was du siehst. Sie sieht Zahlen — und Zahlen kann man täuschen.**

- Sie erkennt Dinge nur mit einer **Wahrscheinlichkeit**, nie sicher.
- Die **Schwelle** legt ein **Mensch** fest — zu hoch, Menschen verschwinden; zu niedrig, Fehlalarme.
- **Nebel und Regen** drücken die Konfidenz unter die Schwelle — die Person ist für die KI weg.

Das Problem ist **nicht gelöst.** Daran wird gearbeitet — vielleicht bald von dir.

---
*Mehr aus diesem Feld: die übrigen Notebooks in diesem Repository — für alle, die tiefer schauen wollen.*
